In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

# ==========================================
# 1. Pipeline Configuration
# ==========================================
# List your 8 sensor IDs exactly as they appear in the CSV filenames/SITE_ID
sensor_ids = ['203', '215', '270', '452', '463', '500', '501', '672'] 

# (Optional) If traffic is one master file for all sensors, load it once here.
# If it's separate files per sensor, move this into the loop.
# df_traffic_master = pd.read_csv('all_traffic_data.csv')

# Dictionary to store our fitted models and cleaned data for later plotting
results_dict = {}

print("Starting MLR Pipeline...\n" + "="*40)

for s_id in sensor_ids:
    print(f"\nAnalyzing Sensor: {s_id}")
    
    # ==========================================
    # 2. Load Data
    # ==========================================
    # Load the specific tree file for this sensor
    tree_file = f'tree_output_{s_id}.csv'
    if not os.path.exists(tree_file):
        print(f"  -> File {tree_file} not found. Skipping.")
        continue
    df_tree = pd.read_csv(tree_file)
    
    # Load or filter the traffic data
    df_traffic = pd.read_csv(f'../Traffic Data/pollution_{s_id}.csv') 
    # OR if using a master traffic file:
    # df_traffic = df_traffic_master[df_traffic_master['SITE_ID'] == int(s_id)].copy()
    
    # ==========================================
    # 3. Temporal Alignment (The Timezone Fix)
    # ==========================================
    # Convert both to Pandas datetime objects
    df_tree['dt_merge'] = pd.to_datetime(df_tree['DATE_TIME_STR'])
    df_traffic['dt_merge'] = pd.to_datetime(df_traffic['DATE_TIME'])
    
    # Force both columns to be "timezone-naive" so they match perfectly
    # .dt.tz_localize(None) strips away the '+00:00' from the traffic data
    # Force both columns to be "timezone-naive" so they match perfectly
    if df_tree['dt_merge'].dt.tz is not None:
        df_tree['dt_merge'] = df_tree['dt_merge'].dt.tz_localize(None)
    if df_traffic['dt_merge'].dt.tz is not None:
        df_traffic['dt_merge'] = df_traffic['dt_merge'].dt.tz_localize(None)
        
    # Merge on the aligned datetime column
    df_merged = pd.merge(df_tree, df_traffic, on='dt_merge', how='inner')
    
    # ==========================================
    # 4. Cleaning & NaN Removal
    # ==========================================
    initial_len = len(df_merged)
    # We only need these three columns to run the MLR
    cols_to_check = ['NOX', 'G_h_effective', 'traffic_nox_index']
    
    df_clean = df_merged.dropna(subset=cols_to_check).copy()
    print(f"  -> Data matched/cleaned: {len(df_clean)} valid hours remaining.")
    
    if len(df_clean) == 0:
        print("  -> Error: No overlapping data. Check your timestamps!")
        continue
        
    # ==========================================
    # 5. Standardization (Z-Scores)
    # ==========================================
    # Z = (Value - Mean) / Standard Deviation
    df_clean['Z_trees'] = (df_clean['G_h_effective'] - df_clean['G_h_effective'].mean()) / df_clean['G_h_effective'].std()
    df_clean['Z_traffic'] = (df_clean['traffic_nox_index'] - df_clean['traffic_nox_index'].mean()) / df_clean['traffic_nox_index'].std()
    
    # ==========================================
    # 6. Multiple Linear Regression (MLR)
    # ==========================================
    # Define Independent Variables (X) and Target Variable (y)
    X = df_clean[['Z_traffic', 'Z_trees']]
    y = df_clean['NOX']
    
    # Add the intercept (beta_0) - Required for Statsmodels
    X = sm.add_constant(X)
    
    # Fit the model
    model = sm.OLS(y, X).fit()
    
    # Extract Metrics
    b0 = model.params['const']
    b_traffic = model.params['Z_traffic']
    b_trees = model.params['Z_trees']
    p_trees = model.pvalues['Z_trees']
    r_sq = model.rsquared
    
    # Print a clean, academic summary for this sensor
    print(f"  -> R-squared: {r_sq:.4f}")
    print(f"  -> Background NOx (Intercept): {b0:.2f} µg/m³")
    print(f"  -> Traffic Coefficient:  +{b_traffic:.4f} (p={model.pvalues['Z_traffic']:.3e})")
    print(f"  -> Tree Coefficient:     {b_trees:.4f} (p={p_trees:.3e})")
    
    if b_trees < 0 and p_trees < 0.05:
        print("  >>> SUCCESS: Significant tree sink effect detected! <<<")
    elif b_trees < 0:
        print("  >>> Note: Tree effect is negative, but not statistically significant.")
    else:
        print("  >>> Warning: Tree coefficient is positive.")

    # Save residuals for later analysis/plotting
    df_clean['residuals'] = model.resid
    
    # Store everything in the dictionary
    results_dict[s_id] = {
        'model': model,
        'data': df_clean,
        'summary': model.summary()
    }

# ==========================================
# Optional: Print Full Academic Summary for a specific sensor
# print(results_dict['203']['summary'])

Starting MLR Pipeline...

Analyzing Sensor: 203
  -> Data matched/cleaned: 8526 valid hours remaining.
  -> R-squared: 0.0167
  -> Background NOx (Intercept): 32.75 µg/m³
  -> Traffic Coefficient:  +4.9454 (p=3.837e-33)
  -> Tree Coefficient:     0.7296 (p=7.562e-02)
  >>> Warning: Tree coefficient is positive.

Analyzing Sensor: 215
  -> Data matched/cleaned: 8483 valid hours remaining.
  -> R-squared: 0.0030
  -> Background NOx (Intercept): 50.19 µg/m³
  -> Traffic Coefficient:  +2.1625 (p=9.090e-04)
  -> Tree Coefficient:     -3.0200 (p=3.635e-06)
  >>> SUCCESS: Significant tree sink effect detected! <<<

Analyzing Sensor: 270
  -> Data matched/cleaned: 8734 valid hours remaining.
  -> R-squared: 0.0177
  -> Background NOx (Intercept): 56.73 µg/m³
  -> Traffic Coefficient:  +6.7425 (p=1.017e-35)
  -> Tree Coefficient:     0.1517 (p=7.780e-01)
  >>> Warning: Tree coefficient is positive.

Analyzing Sensor: 452
  -> Data matched/cleaned: 8644 valid hours remaining.
  -> R-squared: 0.0